In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Análisis exploratorio

In [2]:
df = pd.read_csv("adult.csv", na_values=["?"], skipinitialspace=True)

In [3]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
print(df.size)
print(df.shape)

732630
(48842, 15)


In [5]:
df.isna().sum()

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64

# Preprocesamiento antes de data splitting

In [6]:
# Eliminar variables duplicadas y no relevantes
df = df.drop(columns=["educational-num", "fnlwgt"])

In [7]:
# Reemplazar valores faltantes por "Unknown"
df["workclass"] = df["workclass"].fillna("Unknown")
df["occupation"] = df["occupation"].fillna("Unknown")
df["native-country"] = df["native-country"].fillna("Unknown")

In [9]:
# separarar features y target
X = df.drop("income", axis=1)
y = df["income"].squeeze()

# quitar "." de valores como "<=50K." y ">50K."
y = y.str.replace(".", "", regex=False)
# convertir strings a 0 y 1 (binario)
y = y.map({"<=50K": 0, ">50K": 1})

In [15]:
y.value_counts()

income
0    37155
1    11687
Name: count, dtype: int64

In [18]:
X

,age,workclass,education,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,Unknown,Some-college,Never-married,Unknown,Own-child,White,Female,0,0,30,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States
48838,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States
48839,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States
48840,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States


# Data splitting

In [19]:
from sklearn.model_selection import train_test_split
# Separar en dataset de training y dataset de testing
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Preprocesamiento después de data splitting

In [20]:
# Identificar variables numéricas y categóricas
categorical_cols = X_train_full.select_dtypes(include=["object", "string", "category"]).columns
numerical_cols = X_train_full.select_dtypes(exclude=["object", "string", "category"]).columns

print("Categorical columns:", list(categorical_cols))
print("Numerical columns:", list(numerical_cols))

Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']
Numerical columns: ['age', 'capital-gain', 'capital-loss', 'hours-per-week']


In [21]:
# Escalado necesario para variables numéricas y one-hot encoding para variables categóricas
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
    ]
)